In [1]:
import polars as pl

df = pl.read_csv(
    "../data/megascale.csv",
    null_values=["NA"],
    infer_schema_length=10000
)

columns_to_convert = ['dG_ML', 'ddG_ML']

df = df.with_columns(
    pl.col(columns_to_convert)
    .str.replace_all(r"[^0-9.-]", "")
    .cast(pl.Float64, strict=False)
)

df = df.select([
    "WT_name",
    "name",
    "deltaG",
    "mut_type",
    "aa_seq",
    "aa_seq_full",
])

df = df.rename(
    {
        'aa_seq': 'mutated_seq_short',
        'aa_seq_full': 'mutated_seq_full'
    })

df

WT_name,name,deltaG,mut_type,mutated_seq_short,mutated_seq_full
str,str,f64,str,str,str
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb""",3.332126,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wtm""",3.399832,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wte""",3.315782,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wty""",3.272933,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb_wth""",2.570242,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGSAGGDEVTIHLGDKTIRV…"
…,…,…,…,…,…
"""9AME.pdb""","""9AME.pdb_dmutv5_32I:40L_I32P:L…",-2.85014,"""I32P:L40I""","""NQASVVANQLIPINTALTLVMMRSEVVTPV…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…"
"""9AME.pdb""","""9AME.pdb_dmutv5_32I:40L_I32P:L…",-3.924992,"""I32P:L40W""","""NQASVVANQLIPINTALTLVMMRSEVVTPV…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…"
"""9AME.pdb""","""9AME.pdb_dmutv5_32I:40L_I32P:L…",-2.365774,"""I32P:L40Y""","""NQASVVANQLIPINTALTLVMMRSEVVTPV…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…"


In [2]:
import polars as pl
import re
import polars as pl

df_not_wt = df.filter(pl.col('mut_type') != "wt")

MUTATION_REGEX = re.compile(r"([A-Z])(\d+)([A-Z])")

def reconstruct_original_sequence(
    mutated_seq_short: str,
    mut_type: str,
    mutated_seq_full: str
) -> str:
    if not mut_type or "wt" in mut_type.lower():
        return "-"

    index_offset = mutated_seq_full.find(mutated_seq_short)
    if index_offset == -1:
        return "-"

    mutations = mut_type.split(":")
    reverted_seq = mutated_seq_full


    for mutation in mutations:
        match = MUTATION_REGEX.match(mutation)
        if not match:
            # Formát mutace je neplatný
            return "-"

        original_aa, position_str, mutated_aa = match.groups() # Novou (mutovanou) AMK nepotřebujeme

        if original_aa == mutated_aa:
            continue
        position = int(position_str)

        # Výpočet absolutního 0-based indexu v plné sekvenci
        # pozice - 1 (pro 0-based index) + posun
        absolute_index = position - 1 + index_offset

        # Ošetření, pokud je index mimo platný rozsah sekvence
        if not (0 <= absolute_index < len(reverted_seq)):
            return "-"

        # Provedení reverze: na daném indexu nahradíme aminokyselinu tou původní.
        reverted_seq = (
            reverted_seq[:absolute_index] +
            original_aa +
            reverted_seq[absolute_index + 1:]
        )

    if reverted_seq == mutated_seq_full:
        return "-"
    return reverted_seq


df_not_wt = df_not_wt.with_columns(
    pl.struct(['mut_type', 'mutated_seq_full', "mutated_seq_short"])
    .map_elements(
        lambda row: reconstruct_original_sequence(row['mutated_seq_short'], row['mut_type'], row['mutated_seq_full']),
        return_dtype=pl.String)
    # Výsledek uložíme do nového sloupce 'mutated_seq'
    .alias('original_seq_full'),

)

df_not_wt = df_not_wt.select(['name', 'WT_name', 'mut_type', 'mutated_seq_full', 'original_seq_full', "deltaG", "mutated_seq_short"])

# Přejmenování sloupce 'aa_seq' pro lepší srozumitelnost

df_not_wt

name,WT_name,mut_type,mutated_seq_full,original_seq_full,deltaG,mutated_seq_short
str,str,str,str,str,f64,str
"""EA|run2_0325_0005.pdb_D1Q""","""EA|run2_0325_0005.pdb""","""D1Q""","""SAGGSAGGSAGGQEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.242632,"""QEVTIHLGDKTIRVDGLDKELLEILKELAR…"
"""EA|run2_0325_0005.pdb_D1E""","""EA|run2_0325_0005.pdb""","""D1E""","""SAGGSAGGSAGGEEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.247267,"""EEVTIHLGDKTIRVDGLDKELLEILKELAR…"
"""EA|run2_0325_0005.pdb_D1N""","""EA|run2_0325_0005.pdb""","""D1N""","""SAGGSAGGSAGGNEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.269113,"""NEVTIHLGDKTIRVDGLDKELLEILKELAR…"
"""EA|run2_0325_0005.pdb_D1H""","""EA|run2_0325_0005.pdb""","""D1H""","""SAGGSAGGSAGGHEVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.234823,"""HEVTIHLGDKTIRVDGLDKELLEILKELAR…"
"""EA|run2_0325_0005.pdb_D1R""","""EA|run2_0325_0005.pdb""","""D1R""","""SAGGSAGGSAGGREVTIHLGDKTIRVDGLD…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",2.912039,"""REVTIHLGDKTIRVDGLDKELLEILKELAR…"
…,…,…,…,…,…,…
"""9AME.pdb_dmutv5_32I:40L_I32P:L…","""9AME.pdb""","""I32P:L40I""","""SAGGSAGGNQASVVANQLIPINTALTLVMM…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…",-2.85014,"""NQASVVANQLIPINTALTLVMMRSEVVTPV…"
"""9AME.pdb_dmutv5_32I:40L_I32P:L…","""9AME.pdb""","""I32P:L40W""","""SAGGSAGGNQASVVANQLIPINTALTLVMM…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…",-3.924992,"""NQASVVANQLIPINTALTLVMMRSEVVTPV…"
"""9AME.pdb_dmutv5_32I:40L_I32P:L…","""9AME.pdb""","""I32P:L40Y""","""SAGGSAGGNQASVVANQLIPINTALTLVMM…","""SAGGSAGGNQASVVANQLIPINTALTLVMM…",-2.365774,"""NQASVVANQLIPINTALTLVMMRSEVVTPV…"


In [3]:
df_wt = df.filter(pl.col('name') == pl.col('WT_name')).filter(pl.col('mut_type') == 'wt')
df_wt

WT_name,name,deltaG,mut_type,mutated_seq_short,mutated_seq_full
str,str,f64,str,str,str
"""EA|run2_0325_0005.pdb""","""EA|run2_0325_0005.pdb""",3.332126,"""wt""","""DEVTIHLGDKTIRVDGLDKELLEILKELAR…","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…"
"""EA|run3_0321_0005.pdb""","""EA|run3_0321_0005.pdb""",3.322303,"""wt""","""HDVTIHAGDKTIHVHGASEEFLRIIEQAKR…","""SAGGSAGGSAGGHDVTIHAGDKTIHVHGAS…"
"""EA|run3_0525_0006.pdb""","""EA|run3_0525_0006.pdb""",3.717377,"""wt""","""SHFELRVGTITLHFDNISEELAEELEKLAK…","""SAGGSAGGSAGGSHFELRVGTITLHFDNIS…"
"""EA|run3_1140_0005.pdb""","""EA|run3_1140_0005.pdb""",3.926922,"""wt""","""FHVTIHVGDITFHIHGVSEEEVKKLEELVR…","""SAGGSAGGSAGGFHVTIHVGDITFHIHGVS…"
"""EA|run5_0050_0004.pdb""","""EA|run5_0050_0004.pdb""",4.381241,"""wt""","""TEVDLHLGDITIKLKDVSEEIVKRAKELFK…","""SAGGSAGGSAGGTEVDLHLGDITIKLKDVS…"
…,…,…,…,…,…
"""2KT8.pdb""","""2KT8.pdb""",5.364639,"""wt""","""AEKTGIVNVSSSLNVREGASTSSKVIGSLS…","""SAGGSAAEKTGIVNVSSSLNVREGASTSSK…"
"""2KRS.pdb""","""2KRS.pdb""",6.64727,"""wt""","""MQGVVKVNSALNMRSGPGSNYGVIGTLRNN…","""SAGGMQGVVKVNSALNMRSGPGSNYGVIGT…"
"""2KYB.pdb""","""2KYB.pdb""",4.953675,"""wt""","""TGIVNVSSSLNVRSSASTSSKVIGSLSGNT…","""SAGGSAGGSAGTGIVNVSSSLNVRSSASTS…"


In [4]:
df_wt = df_wt.select(['name', 'mutated_seq_full', 'deltaG'])

df_wt = df_wt.rename(
    {
        'mutated_seq_full': 'original_seq_full',
    }
)

df = df.filter(pl.col('name') != pl.col('WT_name')).filter(pl.col('mut_type') != 'wild_type')

df_wt

name,original_seq_full,deltaG
str,str,f64
"""EA|run2_0325_0005.pdb""","""SAGGSAGGSAGGDEVTIHLGDKTIRVDGLD…",3.332126
"""EA|run3_0321_0005.pdb""","""SAGGSAGGSAGGHDVTIHAGDKTIHVHGAS…",3.322303
"""EA|run3_0525_0006.pdb""","""SAGGSAGGSAGGSHFELRVGTITLHFDNIS…",3.717377
"""EA|run3_1140_0005.pdb""","""SAGGSAGGSAGGFHVTIHVGDITFHIHGVS…",3.926922
"""EA|run5_0050_0004.pdb""","""SAGGSAGGSAGGTEVDLHLGDITIKLKDVS…",4.381241
…,…,…
"""2KT8.pdb""","""SAGGSAAEKTGIVNVSSSLNVREGASTSSK…",5.364639
"""2KRS.pdb""","""SAGGMQGVVKVNSALNMRSGPGSNYGVIGT…",6.64727
"""2KYB.pdb""","""SAGGSAGGSAGTGIVNVSSSLNVRSSASTS…",4.953675


In [5]:

# df_joined = df_not_wt.join(df_wt, left_on='WT_name', right_on='name', how='left', suffix='_wt')
df_joined = df_not_wt.join(df_wt, left_on='original_seq_full', right_on='original_seq_full', how='left', suffix='_wt')

# group by mut_type and avg the dg
df_joined = df_joined.group_by("mutated_seq_full").agg(
    pl.col("deltaG").mean().alias("deltaG"),
    pl.col("deltaG_wt").mean().alias("deltaG_wt"),
    pl.col("original_seq_full").first().alias("original_seq_full"),
    #  pl.col("original_seq_full_wt").first().alias("original_seq_full_wt"),
    pl.col("mut_type").first().alias("mut_type"),
)

df_joined = df_joined.with_columns(
    (pl.col('deltaG') - pl.col('deltaG_wt')).alias('ddG')
)

df_joined.drop_nulls(subset=['ddG'])

df_joined = df_joined.select([

    # 'original_seq_full_wt',

    'original_seq_full', 'mutated_seq_full', 'deltaG', 'deltaG_wt', 'ddG', 'mut_type'
])

df_joined = df_joined.drop_nulls(["ddG"])
df_joined = df_joined.filter(pl.col('original_seq_full') != '-')


# reverse mutation
df_joined = df_joined.with_columns(pl.lit(False).alias("reverse"))

# Reverzní mutace
df_joined_with_reverse = df_joined.with_columns([
    pl.col("original_seq_full").alias("mutated_seq_full"),
    pl.col("mutated_seq_full").alias("original_seq_full"),
    # multiply by -1
    pl.col("ddG") * -1,

    pl.lit(True).alias("reverse")
])

df_joined_with_reverse = pl.concat([df_joined, df_joined_with_reverse])

df_joined_with_reverse.write_csv("datasets/megascale_dataset_with_ddg.csv")

df_joined_with_reverse


original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type,reverse
str,str,f64,f64,f64,str,bool
"""SAGGTYTWNTKEEAKQAFKELLKEKRVPSN…","""SAGGTYTWNTKEEAKQAFKELLKEKPVPSN…",0.609712,2.23667,-1.626958,"""R22P""",false
"""SAGGSAGGSAGGHEITLHINGRRVKLRFRD…","""SAGGSAGGSAGGHEITLHINGRRVKLRFTD…",3.58818,4.440977,-0.852797,"""R17T""",false
"""SAGGSAGGSKDPKFEAAYDFPGSGSSSELP…","""SAGGSAGGSKDPKFEAAMDFPGSGSSSELP…",1.333048,2.501842,-1.168794,"""Y9M:K24Q""",false
"""SAGGNKASVVANQLIPINTALTLIMMKAEV…","""SAGGNKASVVANQLIPINTALTLIMMKAEV…",3.863437,4.161053,-0.297616,"""K39N""",false
"""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…","""SAGGSAGWVPTKREEKYGVAFYNYDARGAD…",0.75604,2.212657,-1.456617,"""D31M:T47G""",false
…,…,…,…,…,…,…
"""SAGGNQASVVANQLIPINTALTLVMMRSEE…","""SAGGNQASVVANQLIPINTALTLVMMRSEV…",7.052293,6.031509,-1.020783,"""V26E""",true
"""SAGGSAGGSAQGDIVVALYPEDGIHPDDLS…","""SAGGSAGGSAQGDIVVALYPYDGIHPDDLS…",-1.42882,3.932004,5.360824,"""Y11E:Y54I""",true
"""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…","""SAGGSAGGMTYKLILNGKTLKGETTTEAVD…",1.998289,2.698998,0.700709,"""D47K""",true


In [6]:
# join back and campute ddg (just the difference)

# join WT_name to name in df_wt

df_joined = df_not_wt.join(df_wt, left_on='WT_name', right_on='name', how='left', suffix='_wt')
# df_joined = df_not_wt.join(df_wt, left_on='original_seq_full', right_on='original_seq_full', how='left', suffix='_wt')


# group by mut_type and avg the dg
df_joined = df_joined.group_by("mutated_seq_full").agg(
    pl.col("deltaG").mean().alias("deltaG"),
    pl.col("deltaG_wt").mean().alias("deltaG_wt"),
    pl.col("original_seq_full").first().alias("original_seq_full"),
    pl.col("original_seq_full_wt").first().alias("original_seq_full_wt"),
    pl.col("mut_type").first().alias("mut_type"),
)

df_joined = df_joined.with_columns(
    (pl.col('deltaG') - pl.col('deltaG_wt')).alias('ddG')
)

df_joined.drop_nulls(subset=['ddG'])

df_joined = df_joined.select([

    'original_seq_full_wt',
    'original_seq_full', 'mutated_seq_full', 'deltaG', 'deltaG_wt', 'ddG', 'mut_type'
])

df_joined.filter(pl.col('mut_type') != 'wild_type').filter(
    pl.col('original_seq_full_wt') == pl.col('original_seq_full'))

df_joined = df_joined.drop_nulls(["ddG"])
df_joined = df_joined.filter(pl.col('original_seq_full') != '-')


df_joined


original_seq_full_wt,original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type
str,str,str,f64,f64,f64,str
"""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRE…","""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRE…","""SAGGSPAHPYDRLKTTSTDPVSDIDVTRRG…",0.611124,1.254028,-0.642904,"""E25G:Y45W"""
"""SAGGSAGGSAGGYNLQKLLAPYHKAKTLER…","""SAGGSAGGYNLQKLLAPYHKAKTLERQVYE…","""SAGGSAGGYNLQKLLAPYHKAKTLERQVYE…",-0.00611,2.057796,-2.063906,"""E24G:R46S"""
"""GGSQTQRTQDENEARRIAEEWKRRGYDVEV…","""GGSQTQRTQDENEARRIAEEWKRRGYDVEV…","""GGSQTQRTQDENEARRIAEEWKRRGYDVEV…",2.468309,2.827459,-0.35915,"""R29A"""
"""SAGGMIINNLKLIREKKKISQSELAALLES…","""SAGGSAGGMIINNLKLIREKKKISQSELAA…","""SAGGSAGGMIINNLKLIREKKKISCSELAA…",2.025331,3.380225,-1.354895,"""Q17C"""
"""NNDALSPAIRRLLAEHNLDASAIKGTGVGG…","""NNDALSPAIRRLLAEHNLDASAIKGTGVGG…","""NNIALSPAIRRLLAEHNLDASAIKGTGVGG…",2.759837,2.996319,-0.236482,"""D3I"""
…,…,…,…,…,…,…
"""SAGGSAGGSAGGTTYKLILNGKTLKGETTT…","""SAGGSAGGSAGGTTYKLILNGKTLKGETTT…","""SAGGSAGGSAGGTTYKGILNGKTLKGETTT…",-0.78508,5.632582,-6.417662,"""L5G:F30T"""
"""SAGGSMIINNLKLIREKKKISQSELAALLE…","""SAGGSMIINNLKLIREKKKISQSELAALLE…","""SAGGSMIINNLKEIREKKKISQSELAALLE…",3.80393,3.82955,-0.02562,"""L8E"""
"""SAGGSAGGSAGGSAGGMISNAKIARINELA…","""SAGGSAGGSAGGSAGGMISNAKIARINELA…","""SAGGSAGGSAGGSAGGMISNAKIAWINELA…",0.481303,2.215058,-1.733755,"""R9W:E29F"""


In [8]:
df_joined_with_reverse.filter(pl.col('original_seq_full') == pl.col('mutated_seq_full'))

# df_joined_with_reverse.filter(pl.col('mut_type').str.contains(":")).filter(pl.col('original_seq_full') == pl.col('mutated_seq_full'))

original_seq_full,mutated_seq_full,deltaG,deltaG_wt,ddG,mut_type,reverse
str,str,f64,f64,f64,str,bool
